# Notebook D — SSVI Surface Risk Engine

**SSVI Volatility Surface Dynamics — S&P 500 Options (2010–2020)**
Politecnico di Milano — Econometrics Project (A.Y. 2025/26)

---

## Overview

This notebook builds a **volatility-surface risk engine** for option market-making
using daily SPX options data from notebooks A/B/C as inputs.

**Research question:**
> *Can localized realized volatility of the SSVI implied-volatility surface improve
> volatility-risk-adjusted option market-making spreads beyond a baseline
> Avellaneda-style quoting rule?*

**Pipeline:**
```
data/ssvi_all_dates_clean_results.csv          ← SSVI calibration (notebook B)
  → IV surface reconstruction (45 grid points)
  → ΔIV daily first differences
  → surface_move (equal-weighted RMS)
  → HAR-J RV forecast (train-only OLS)
  → c*(ES₉₅, val-set) calibration
  → spread_final = spread_AS + c* × RV̂
  → test-set backtest (coverage, vega robustness)
```

**Methodological constraints:**
- PCA fitted on **train set only**; projected onto val/test
- `c*` calibrated on **validation set only**; evaluated on test set
- HMM is **descriptive**; Viterbi path is retrospective
- No cap on market spread; no shuffle in train/val/test splits
- Baseline spread is an Avellaneda-*style* proxy (not the full AS model)

**Sections:**
1. Setup  ·  2. Data Diagnostic  ·  3. IV Surface  ·  4. Surface RV & Splits
5. HAR Forecasting  ·  6. PCA  ·  7. HMM Regimes  ·  8. Spread Engine
9. c* Term Structure  ·  10. Backtest  ·  11. Vega Robustness  ·  12. Summary


## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as smapi

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Path resolution (works from econometric_analysis/, notebook/, or repo root) ──
BASE = Path.cwd().resolve()
if BASE.name in ('econometric_analysis', 'notebook'):
    BASE = BASE.parent

SRC_DIR    = BASE / 'src'
DATA_DIR   = BASE / 'data'
OUTPUT_DIR = BASE / 'output'
PLOT_DIR   = OUTPUT_DIR / 'plots'
CACHE_DIR  = OUTPUT_DIR / 'cache'

for _d in (OUTPUT_DIR, PLOT_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ssvi_mm_risk_engine import (
    get_grid_metadata, load_ssvi_results, build_iv_panel,
    compute_delta_iv_panel, compute_surface_move, compute_move_by_maturity,
    build_future_rv_targets, chronological_split, ensure_dirs,
    make_har_features, detect_jumps_bpv, choose_jump_method,
    fit_har_model, qlike_loss, diebold_mariano_test, compute_forecast_metrics,
    fit_train_pca, transform_pca_splits, save_pca_outputs,
    fit_hmm_regimes, summarize_hmm_regimes,
    compute_avellaneda_style_spread, expected_shortfall, calibrate_c_es95,
    compute_spread_final, backtest_spread_coverage, compute_vega_weighted_robustness,
    plot_surface_move, plot_move_by_maturity, plot_jump_detection_comparison,
    plot_pca_loadings, plot_pca_scores, plot_hmm_regimes, plot_forecast_global,
    plot_cstar_term_structure, plot_spread_decomposition,
)

NB = 'D'   # output-file prefix for all saved files

# ── Grid ──────────────────────────────────────────────────────────────────────
K_GRID = np.array([-0.30, -0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.20, 0.30])
T_GRID = np.array([1/12, 3/12, 6/12, 1.0, 2.0])
k_flat, t_flat, GRID_COLS, T_LABELS, MAT_T_MAP = get_grid_metadata(K_GRID, T_GRID)

BUCKET_COLS = {
    'left_wing':  [c for c, kv in zip(GRID_COLS, k_flat) if kv <= -0.10],
    'atm':        [c for c, kv in zip(GRID_COLS, k_flat) if abs(kv) <= 0.05],
    'right_wing': [c for c, kv in zip(GRID_COLS, k_flat) if kv >= 0.10],
}

# ── Strategy parameters ───────────────────────────────────────────────────────
GAMMA_AS    = 0.50   # risk-aversion proxy (heuristic; uncalibrated)
KAPPA_K     = 0.50   # wing-widening factor
DT          = 1 / 252
CALIB_ALPHA = 0.95
HORIZONS    = [5, 10, 22]
H_MAIN      = 5

print(f'Backend   : {matplotlib.get_backend()}')
print(f'BASE      : {BASE}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'Grid      : {len(K_GRID)} k × {len(T_GRID)} T = {len(GRID_COLS)} points')


## 2. Data Diagnostic: Calibration Exclusions

The raw SSVI results file may contain failed calibration dates (typically early in the
sample when the options chain had too few clean observations).
We document excluded dates before building the surface.

In [ ]:
import re as _re

_raw = pd.read_csv(DATA_DIR / 'ssvi_all_dates_clean_results.csv')
_failed = _raw[_raw['n_obs'] == 0].copy()
_failed['actual_n'] = _failed['message'].apply(
    lambda m: int(x.group(1)) if (x := _re.search(r'Only (\d+) obs', str(m))) else None
)

print('=== CALIBRATION EXCLUSION DIAGNOSTIC ===')
print(f'Total dates in CSV      : {len(_raw)}')
print(f'Successful calibrations : {(_raw["n_obs"] > 0).sum()}')
print(f'Failed (n_obs = 0)      : {len(_failed)}  ({len(_failed)/len(_raw)*100:.1f}%)')
print()
print('Failed dates by time_elapsed range:')
_bins  = [0, 50, 100, 200, 500, 5000]
_lbls  = ['0-50', '50-100', '100-200', '200-500', '500+']
_failed['te_bucket'] = pd.cut(_failed['time_elapsed'], bins=_bins, labels=_lbls)
print(_failed['te_bucket'].value_counts().sort_index().to_string())
print()
print('Actual obs in failed rows:')
print(f'  min={_failed["actual_n"].min()}  median={_failed["actual_n"].median():.0f}  max={_failed["actual_n"].max()}')
print(f'  ≥80 obs: {(_failed["actual_n"]>=80).sum()}  |  60-79 obs: {((_failed["actual_n"]>=60)&(_failed["actual_n"]<80)).sum()}  |  <60 obs: {(_failed["actual_n"]<60).sum()}')
print()
print('Threshold sensitivity (how many dates recovered if threshold were lowered):')
for _t in [40, 60, 80]:
    _r = (_failed['actual_n'] >= _t).sum()
    print(f'  threshold = {_t}: +{_r} dates ({_r/len(_raw)*100:.1f}% of dataset)')
print()
print('RECOMMENDATION: Keep threshold at 100.')
print('  The first ~103 trading days (≈ Jan–May 2010) are systematically sparse (~58 obs).')
print('  The 47 borderline dates (80-97 obs) are scattered anomalies; excluding them is conservative.')
print('  Coverage-based alternative: n_obs ≥ 60 AND balanced smile (≥5 left, ≥5 right, ≥3 ATM).')


## 3. IV Surface Reconstruction

SSVI calibration parameters are loaded from `data/`.
The IV panel (45-point grid) is loaded from `output/ssvi_surface_iv_panel.csv` if cached,
otherwise reconstructed from SSVI parameters.

Safety checks confirm DatetimeIndex, monotone ordering, and A→D index alignment.

In [ ]:
# ── Load calibration parameters (DATA_DIR = data/) ────────────────────────────
params_ok = load_ssvi_results(DATA_DIR, OUTPUT_DIR)

# ── Safety assertions ─────────────────────────────────────────────────────────
assert isinstance(params_ok.index, pd.DatetimeIndex), \
    'params_ok.index must be DatetimeIndex — check load_ssvi_results / time_elapsed column'
assert params_ok.index.is_monotonic_increasing, \
    'params_ok.index is not monotonically increasing — check for gaps or duplicates'
assert not params_ok.index.duplicated().any(), \
    f'{params_ok.index.duplicated().sum()} duplicate dates found in params_ok'

print(f'Successful calibrations : {len(params_ok)}')
print(f'Date range              : {params_ok.index[0].date()} → {params_ok.index[-1].date()}')
_p = [c for c in ('alpha', 'beta', 'rho', 'eta', 'gamma') if c in params_ok.columns]
print(params_ok[_p].describe().round(4).to_string())

# ── Reconstruct IV panel ───────────────────────────────────────────────────────
# Load from cache if it exists; rebuild and cache if not.
_iv_cache = OUTPUT_DIR / 'ssvi_surface_iv_panel.csv'
iv_panel = build_iv_panel(
    params_ok, k_flat, t_flat, GRID_COLS,
    src_path = _iv_cache,
    out_path = _iv_cache if not _iv_cache.exists() else OUTPUT_DIR / f'{NB}_iv_panel.csv',
)

assert isinstance(iv_panel.index, pd.DatetimeIndex), \
    'iv_panel.index must be DatetimeIndex — check build_iv_panel / cache file'
assert iv_panel.index.is_monotonic_increasing, 'iv_panel.index is not sorted'

print(f'\nIV panel : {iv_panel.shape}  NaN={iv_panel.isnull().mean().mean()*100:.2f}%')
print(f'Dates    : {iv_panel.index[0].date()} → {iv_panel.index[-1].date()}')
print(f'ATM 3M mean IV: {iv_panel["iv_k_0.00_T_0.25"].mean():.4f}')


## 4. Surface Realized Volatility & Train/Val/Test Splits

`ΔIV(k,T,t) = IV(k,T,t) − IV(k,T,t−1)`.
`surface_move(t)` = equal-weighted RMS of ΔIV across all 45 grid points.
Targets: `future_rv_h(t) = √( mean_{j=1}^h smove(t+j)² )` — forward-looking only, no leakage.
Splits: 70% train / 15% val / 15% test (chronological, no shuffle).

In [ ]:
delta_iv    = compute_delta_iv_panel(iv_panel)
smove       = compute_surface_move(delta_iv)
move_by_mat = compute_move_by_maturity(delta_iv, GRID_COLS, T_LABELS, MAT_T_MAP)
targets_df  = build_future_rv_targets(smove, move_by_mat, HORIZONS, T_LABELS)

pd.concat([smove, move_by_mat], axis=1).to_csv(OUTPUT_DIR / f'{NB}_surface_moves.csv')
print(f'delta_iv    : {delta_iv.shape}')
print(f'surface_move: mean={smove.mean():.5f}  std={smove.std():.5f}')
print(f'targets     : {targets_df.shape}  non-NaN 5d: {targets_df["future_rv_5d"].notna().sum()}')

plot_surface_move(smove, PLOT_DIR, prefix=NB)
plot_move_by_maturity(move_by_mat, PLOT_DIR, prefix=NB)

split_train, split_val, split_test = chronological_split(smove.index, 0.70, 0.15)
print(f'Train : {len(split_train)}  [{split_train[0].date()} → {split_train[-1].date()}]')
print(f'Val   : {len(split_val)}   [{split_val[0].date()}  → {split_val[-1].date()}]')
print(f'Test  : {len(split_test)}   [{split_test[0].date()}  → {split_test[-1].date()}]')

# ── Leakage guard ─────────────────────────────────────────────────────────────
assert split_train[-1] < split_val[0],  'Train/val overlap detected'
assert split_val[-1]   < split_test[0], 'Val/test overlap detected'


## 5. HAR Features & Forecasting

**Jump detection:** BPV baseline (~31% rate) is replaced by a rolling-threshold method
targeting 5–15%.  Method A = rolling q₉₅+2σ; Method B = rolling q₉₉.
Auto-selector picks whichever falls in [5%, 15%], closest to 10%.

**Models:** Persistence · HAR-RV · HAR-J · HAR-CJ — all fitted on **train set only**.
**Evaluation:** MAE, RMSE, QLIKE on log-scale; Diebold-Mariano test vs HAR baseline.

In [ ]:
# ── Initial BPV features (overridden by rolling-threshold below) ──────────────
sm_sq   = smove.pow(2)
j_flag  = detect_jumps_bpv(smove)
feats_g = make_har_features(smove, j_flag=j_flag)

log_rv1  = feats_g['log_rv1'];  log_rv5   = feats_g['log_rv5']
log_rv22 = feats_g['log_rv22']; log_ewma  = feats_g['log_ewma']
j_cnt22  = feats_g['j_cnt22'];  log_j22   = feats_g['log_j22']
log_c5   = feats_g['log_c5'];   log_j5    = feats_g['log_j5']

mat_feats = {}
for t_lbl in T_LABELS:
    ms = move_by_mat.get(f'move_{t_lbl}')
    if ms is not None:
        mat_feats[t_lbl] = make_har_features(ms, j_flag=detect_jumps_bpv(ms))

old_jump_pct = float(j_flag.mean() * 100)
print(f'Jump rate (BPV): {old_jump_pct:.1f}%  → rolling-threshold will select a stricter method.')


In [ ]:
# ── Rolling-threshold jump detection ─────────────────────────────────────────
j_flag_old   = j_flag.copy()
j_flag_new, selected_key, selected_pct, cands = choose_jump_method(smove)
pct_A, pct_B = cands['A'][1], cands['B'][1]
print(f'Jump detection:')
print(f'  BPV (original)    : {old_jump_pct:.1f}%')
print(f'  Method A (q95+2σ) : {pct_A:.1f}%')
print(f'  Method B (q99)    : {pct_B:.1f}%')
print(f'  Selected          : Method {selected_key}  ({selected_pct:.1f}%)')

pd.DataFrame(
    {'bpv_jump': j_flag_old, 'method_A': cands['A'][0],
     'method_B': cands['B'][0], 'selected': j_flag_new},
    index=smove.index,
).to_csv(OUTPUT_DIR / f'{NB}_jump_comparison.csv')

plot_jump_detection_comparison(smove, j_flag_old, j_flag_new,
                                selected_key, old_jump_pct, selected_pct,
                                PLOT_DIR, prefix=NB)

# ── Update features with selected jump method ─────────────────────────────────
j_flag  = j_flag_new.copy()
feats_g = make_har_features(smove, j_flag=j_flag)
log_rv1  = feats_g['log_rv1'];  log_rv5   = feats_g['log_rv5']
log_rv22 = feats_g['log_rv22']; log_ewma  = feats_g['log_ewma']
j_cnt22  = feats_g['j_cnt22'];  log_j22   = feats_g['log_j22']
log_c5   = feats_g['log_c5'];   log_j5    = feats_g['log_j5']

_use_q99 = (selected_key == 'B')
for t_lbl in T_LABELS:
    ms = move_by_mat.get(f'move_{t_lbl}')
    if ms is None or t_lbl not in mat_feats:
        continue
    _rq = ms.rolling(252, min_periods=50).quantile(0.99 if _use_q99 else 0.95)
    _rs = ms.rolling(252, min_periods=50).std()
    j_fl_m = ((ms > _rq).astype(float).fillna(0.0) if _use_q99
              else (ms > (_rq + 2 * _rs)).astype(float).fillna(0.0))
    mat_feats[t_lbl].update(make_har_features(ms, j_flag=j_fl_m))

print(f'\nJump rate: {old_jump_pct:.1f}% → {selected_pct:.1f}%')


In [ ]:
# ── Build feature DataFrame ───────────────────────────────────────────────────
feat_global = pd.DataFrame({
    'log_rv1': log_rv1, 'log_rv5': log_rv5, 'log_rv22': log_rv22,
    'log_j22': log_j22, 'j_cnt22': j_cnt22,
    'log_c5':  log_c5,  'log_j5':  log_j5,
})

LOG_TGT        = 'log_future_rv_5d'
log_tgt_global = np.log(targets_df['future_rv_5d'] + 1e-8).rename(LOG_TGT)

HAR_COLS   = ['log_rv1', 'log_rv5', 'log_rv22']
HARJ_COLS  = ['log_rv1', 'log_rv5', 'log_rv22', 'log_j22', 'j_cnt22']
HARCJ_COLS = ['log_c5',  'log_j5',  'log_rv22']
PERS_COLS  = ['log_rv22']

TARGETS_TO_FIT = {'global': log_tgt_global}
for t_lbl in T_LABELS:
    col = f'future_rv_{H_MAIN}d_{t_lbl}'
    if col in targets_df.columns:
        TARGETS_TO_FIT[t_lbl] = np.log(targets_df[col] + 1e-8).rename(f'log_tgt_{t_lbl}')

all_metrics, fc_test, fc_val = [], {}, {}

for tgt_key, log_tgt in TARGETS_TO_FIT.items():
    fbase = feat_global if tgt_key == 'global' else pd.DataFrame(mat_feats.get(tgt_key, {}))
    if fbase.empty:
        continue
    df_all = pd.concat([log_tgt, fbase], axis=1)
    for mname, fcols in [('Pers',   PERS_COLS),
                          ('HAR',    HAR_COLS),
                          ('HAR-J',  HARJ_COLS),
                          ('HAR-CJ', HARCJ_COLS)]:
        avail = [c for c in fcols if c in df_all.columns]
        if not avail:
            continue
        fc_te = fit_har_model(df_all, avail, log_tgt.name, split_train, split_test)
        fc_va = fit_har_model(df_all, avail, log_tgt.name, split_train, split_val)
        y_te  = log_tgt.reindex(split_test).dropna()
        common = y_te.index.intersection(fc_te.dropna().index)
        if len(common) < 5:
            continue
        m = compute_forecast_metrics(y_te.loc[common].values, fc_te.loc[common].values)
        m.update({'model': mname, 'target': tgt_key})
        all_metrics.append(m)
        fc_test[f'{mname}|{tgt_key}'] = fc_te
        fc_val[f'{mname}|{tgt_key}']  = fc_va

# ── Diebold-Mariano tests ─────────────────────────────────────────────────────
for tgt_key, log_tgt in TARGETS_TO_FIT.items():
    har_fc  = fc_test.get(f'HAR|{tgt_key}')
    harj_fc = fc_test.get(f'HAR-J|{tgt_key}')
    if har_fc is None or harj_fc is None:
        continue
    y_te   = log_tgt.reindex(split_test).dropna()
    common = (y_te.index
              .intersection(har_fc.dropna().index)
              .intersection(harj_fc.dropna().index))
    if len(common) < 20:
        continue
    dm_s, dm_p = diebold_mariano_test(
        y_te.loc[common] - har_fc.loc[common],
        y_te.loc[common] - harj_fc.loc[common], h=H_MAIN)
    for m in all_metrics:
        if m['target'] == tgt_key and m['model'] == 'HAR-J':
            m['DM_stat_vs_HAR'] = dm_s
            m['DM_pval_vs_HAR'] = dm_p

# ── Walk-forward HAR-J (global, refit every 22 days) ─────────────────────────
REFIT_FREQ, MIN_TRAIN = 22, 250
df_wf = pd.concat([log_tgt_global, feat_global[HARJ_COLS]], axis=1).dropna()
fc_wf, _ols_wf, _X_tr = {}, None, None
for i, dt in enumerate(split_test):
    if dt not in df_wf.index:
        continue
    pos = int(np.searchsorted(df_wf.index, dt))
    if pos < MIN_TRAIN:
        continue
    if i % REFIT_FREQ == 0 or _ols_wf is None:
        _X_tr = smapi.add_constant(df_wf.iloc[:pos][HARJ_COLS], has_constant='add')
        _y_tr = df_wf.iloc[:pos][LOG_TGT]
        _ols_wf = smapi.OLS(_y_tr, _X_tr).fit()
    row = smapi.add_constant(df_wf.loc[[dt]][HARJ_COLS], has_constant='add')
    row = row.reindex(columns=_X_tr.columns, fill_value=0.0)
    fc_wf[dt] = float(_ols_wf.predict(row)[0])

fc_wf_ser = pd.Series(fc_wf, name='HAR-J (WF)')
fc_test[f'HAR-J|global|WF'] = fc_wf_ser

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(OUTPUT_DIR / f'{NB}_forecast_metrics.csv', index=False)
pd.DataFrame({k: v for k, v in fc_test.items()}).to_csv(OUTPUT_DIR / f'{NB}_forecasts.csv')

print('=== FORECAST TABLE (global, h=5, test set) ===')
_cols = ['model', 'n', 'MAE_log', 'RMSE_log', 'QLIKE', 'DM_stat_vs_HAR', 'DM_pval_vs_HAR']
print(metrics_df[metrics_df['target'] == 'global']
      [[c for c in _cols if c in metrics_df.columns]].round(4).to_string(index=False))

plot_forecast_global(targets_df, fc_test, fc_wf_ser, split_test, H_MAIN, PLOT_DIR, prefix=NB)


## 6. PCA on ΔIV  (train-only fit)

StandardScaler and PCA are fitted **on the train set only** and then projected onto
val and test sets.  The first three PCs typically capture level, skew, and curvature.

In [ ]:
scaler_pca, pca = fit_train_pca(delta_iv, split_train, n_components=10)
pc_df, loadings_df, ev = transform_pca_splits(
    delta_iv, scaler_pca, pca, split_train, split_val, split_test
)
save_pca_outputs(pc_df, loadings_df, OUTPUT_DIR, prefix=NB)

cum_ev = ev.cumsum()
print(f'PCA (train-only fit) — {pca.n_components_} components')
print(f'PC1-3 cumulative variance: {cum_ev[min(2, len(ev)-1)]*100:.1f}%')
for i in range(min(5, len(ev))):
    print(f'  PC{i+1}: {ev[i]*100:.1f}%  (cum: {cum_ev[i]*100:.1f}%)')

plot_pca_loadings(loadings_df, K_GRID, T_GRID, T_LABELS, ev, PLOT_DIR, prefix=NB)
plot_pca_scores(pc_df, PLOT_DIR, prefix=NB)


## 7. HMM Regime Detection

3-state Gaussian HMM on `log(RV₂₂d)`, fitted on **train+val only**.
Regimes sorted by mean log-RV: 0=low-vol, 1=mid-vol, 2=high-vol.

⚠ The Viterbi path is retrospective and uses the full series for prediction.
This is suitable for **descriptive** labelling only; do not use for live signal generation.

In [ ]:
log_rv22_full = np.log(np.sqrt(sm_sq.rolling(22, min_periods=10).mean()) + 1e-8)
hmm, regime_s, remap, trans_df = fit_hmm_regimes(log_rv22_full, split_train, split_val)

REGIME_LABELS = {0: 'Low-vol', 1: 'Mid-vol', 2: 'High-vol'}
summary_hmm   = summarize_hmm_regimes(regime_s, log_rv22_full, REGIME_LABELS)
regime_s.to_frame().to_csv(OUTPUT_DIR / f'{NB}_hmm_regimes.csv')

print('HMM Regime Summary:')
print(summary_hmm.round(3).to_string(index=False))
print('\nTransition matrix (rows=from, cols=to):')
print(trans_df.round(3).to_string())

plot_hmm_regimes(smove, regime_s, REGIME_LABELS, PLOT_DIR, prefix=NB)


## 8. Spread Engine & c* Calibration

**Baseline proxy** (not the full Avellaneda-Stoikov model):
$$\text{spread}_{AS}(k,T,t) = \gamma \times \sigma_{ATM}(T,t) \times \sqrt{\Delta t} \times (1 + \kappa|k|)$$

`γ=0.5` and `κ=0.5` are uncalibrated heuristic parameters.

**c* calibration** (validation set only):
$$c^*(\text{bucket}) = \text{ES}_{0.95}\!\left(\frac{\max(\text{actual\_rv} - \text{spread}_{AS},\, 0)}{\hat{\text{RV}}}\right)$$

In [ ]:
spread_AS = compute_avellaneda_style_spread(
    iv_panel, GRID_COLS, k_flat, MAT_T_MAP, GAMMA_AS, KAPPA_K, DT
)
print(f'AS spread parameters: GAMMA_AS={GAMMA_AS}  KAPPA_K={KAPPA_K}  DT=1/252')
for bucket, cols in BUCKET_COLS.items():
    mv = float(spread_AS[cols].mean().mean())
    print(f'  {bucket:12s}: {mv:.5f} vol-units')
print(f'  mean surface_move : {smove.mean():.5f} vol-units')


In [ ]:
calib_targets = {'global': ('future_rv_5d', spread_AS.mean(axis=1))}
for t_lbl, t_str in MAT_T_MAP.items():
    tgt_col  = f'future_rv_{H_MAIN}d_{t_lbl}'
    mat_cols = [c for c in GRID_COLS if f'_T_{t_str}' in c]
    if tgt_col in targets_df.columns and mat_cols:
        calib_targets[t_lbl] = (tgt_col, spread_AS[mat_cols].mean(axis=1))

calib_df = calibrate_c_es95(
    split_val, targets_df, calib_targets, fc_val, log_ewma, CALIB_ALPHA
)
calib_df.to_csv(OUTPUT_DIR / f'{NB}_c_es95_calibration.csv')
print('c* calibration (val set, ES₉₅):')
print(calib_df.round(3).to_string())


## 9. c* Term Structure

Plots c*(T) by maturity and fits an OLS regression vs √T.
c* increasing in T → uncertainty accumulates with horizon (standard result).

In [ ]:
T_VALS     = np.array(T_GRID, dtype=float)
cstar_vals = np.array(
    [calib_df.loc[lbl, 'c_star'] if lbl in calib_df.index else np.nan
     for lbl in T_LABELS], dtype=float,
)
cstar_by_mat = pd.DataFrame({
    'maturity': T_LABELS,
    'T':        T_VALS,
    'sqrt_T':   np.sqrt(T_VALS),
    'c_star':   cstar_vals,
    'n_val':    [int(calib_df.loc[lbl, 'n_val']) if lbl in calib_df.index else 0
                 for lbl in T_LABELS],
})
cstar_by_mat.to_csv(OUTPUT_DIR / f'{NB}_cstar_term_structure.csv', index=False)

valid_mask = np.isfinite(cstar_vals)
a_hat = b_hat = r2 = np.nan
sqrtT_fit = cstar_fit = np.array([])
if valid_mask.sum() >= 3:
    sqrtT_v = np.sqrt(T_VALS[valid_mask])
    cstar_v = cstar_vals[valid_mask]
    X_reg   = np.column_stack([np.ones(len(sqrtT_v)), sqrtT_v])
    coeffs, _, _, _ = np.linalg.lstsq(X_reg, cstar_v, rcond=None)
    a_hat, b_hat    = coeffs
    sqrtT_fit = np.linspace(0, np.sqrt(T_VALS.max()) * 1.1, 100)
    cstar_fit = a_hat + b_hat * sqrtT_fit
    ss_res = np.sum((cstar_v - (a_hat + b_hat * sqrtT_v)) ** 2)
    ss_tot = np.sum((cstar_v - cstar_v.mean()) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 1e-15 else np.nan
    print(f'OLS: c*(T) = {a_hat:.3f} + {b_hat:.3f} × √T   (R² = {r2:.3f})')

plot_cstar_term_structure(cstar_by_mat, a_hat, b_hat, r2,
                          sqrtT_fit, cstar_fit, PLOT_DIR, prefix=NB)
print(cstar_by_mat.round(3).to_string(index=False))

if valid_mask.sum() >= 3:
    diffs = np.diff(cstar_vals[valid_mask])
    if np.all(diffs > 0):
        print('\nc* strictly increasing in T — uncertainty accumulates with horizon.')
    elif np.all(diffs < 0):
        print('\nc* strictly decreasing in T — short-dated RV forecasts understate tail risk.')
    else:
        _pk = T_LABELS[np.where(valid_mask)[0][int(np.argmax(cstar_vals[valid_mask]))]]
        print(f'\nc* non-monotone (peak at {_pk}) — bucket-specific calibration warranted.')
    if np.isfinite(r2):
        tag = 'strong' if r2 > 0.80 else ('moderate' if r2 > 0.40 else 'weak')
        print(f'  √T regression fit: {tag} (R²={r2:.2f})')


## 10. Test Set Backtest

```
spread_final(k,T,t) = spread_AS(k,T,t) + c*(bucket) × RV̂(k,T,t)
```

Coverage = P(future_rv_5d ≤ spread_final) — target 95%.
Stacked bar charts show spread decomposition (baseline + addon) by maturity bucket.

In [ ]:
spread_final, addon_panel = compute_spread_final(
    iv_panel, GRID_COLS, MAT_T_MAP, spread_AS, fc_test, calib_df, log_ewma
)
spread_final.to_csv(OUTPUT_DIR / f'{NB}_spread_final.csv')

bt_df = backtest_spread_coverage(
    split_test, calib_targets, targets_df, spread_final, spread_AS,
    grid_cols=GRID_COLS, mat_t_map=MAT_T_MAP,
)
bt_df.to_csv(OUTPUT_DIR / f'{NB}_mm_backtest.csv', index=False)

print('=== MM BACKTEST (test set) ===')
print(bt_df.round(4).to_string(index=False))

plot_spread_decomposition(split_test, targets_df, spread_final, spread_AS, bt_df,
                          PLOT_DIR, h_main=H_MAIN, prefix=NB)


In [ ]:
# ── Main results table ────────────────────────────────────────────────────────
ORDER       = T_LABELS + ['global']
regime_test = regime_s.reindex(split_test)
_rh = float((regime_test == 2).mean()) if len(regime_test) > 0 else np.nan
_rm = float((regime_test == 1).mean()) if len(regime_test) > 0 else np.nan
_rl = float((regime_test == 0).mean()) if len(regime_test) > 0 else np.nan

result_rows = []
for bucket_key in ORDER:
    row_bt = bt_df[bt_df['bucket'] == bucket_key]
    if row_bt.empty:
        continue
    rb       = row_bt.iloc[0].to_dict()
    c_star_v = (float(calib_df.loc[bucket_key, 'c_star'])
                if bucket_key in calib_df.index else np.nan)
    result_rows.append({
        'bucket':            bucket_key,
        'n':                 int(rb['n']),
        'coverage':          rb['coverage'],
        'exceedance':        rb['exceedance'],
        'mean_spread_AS':    rb['mean_AS'],
        'mean_addon':        rb['mean_addon'],
        'mean_spread_final': rb['mean_final'],
        'addon_share':       rb.get('addon_share', np.nan),
        'c_star':            c_star_v,
        'regime_high_pct':   _rh * 100,
        'regime_mid_pct':    _rm * 100,
        'regime_low_pct':    _rl * 100,
    })

results_df = pd.DataFrame(result_rows)
results_df.to_csv(OUTPUT_DIR / f'{NB}_main_results_table.csv', index=False)

print('=== MAIN RESULTS TABLE (test set) ===')
_cols = ['bucket', 'n', 'coverage', 'exceedance', 'mean_spread_AS',
         'mean_addon', 'addon_share', 'c_star']
print(results_df[[c for c in _cols if c in results_df.columns]].round(4).to_string(index=False))

# Stacked bar chart: spread decomposition + coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spread Decomposition by Maturity Bucket (test set)', fontsize=11, fontweight='bold')
x     = np.arange(len(results_df))
lbls  = results_df['bucket'].values
as_v  = results_df['mean_spread_AS'].values
add_v = results_df['mean_addon'].values
cov_v = results_df['coverage'].values
rrs_v = results_df['addon_share'].values

ax = axes[0]
ax.bar(x, as_v,  0.55, label='Baseline AS',  color='steelblue', alpha=0.85)
ax.bar(x, add_v, 0.55, bottom=as_v,          label='Addon c×RV̂', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=9)
ax.set_ylabel('vol-units'); ax.set_title('Stacked Spread Decomposition')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for xi, (a, b) in enumerate(zip(as_v, add_v)):
    total = a + b; share = b / (total + 1e-12) * 100
    ax.text(xi, total + max(total * 0.02, 1e-5),
            f'{share:.0f}%', ha='center', va='bottom', fontsize=8,
            color='darkorange', fontweight='bold')

ax = axes[1]
ax.bar(x - 0.20, cov_v, 0.35, label='Coverage',    color='seagreen',   alpha=0.85)
ax.bar(x + 0.20, rrs_v, 0.35, label='Addon share', color='darkorange', alpha=0.85)
ax.axhline(0.95, ls='--', color='red', lw=1.1, label='95% target')
ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=9)
ax.set_title('Coverage vs Addon Share'); ax.legend(fontsize=9)
ax.set_ylim(0, 1.1); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / f'{NB}_spread_term_structure.png', dpi=150, bbox_inches='tight')
plt.show()

high_addon = results_df[results_df['addon_share'] > 0.5]
if len(high_addon) > 0:
    print(f'\n{len(high_addon)} bucket(s) where addon > 50% of total spread:')
    for _, r in high_addon.iterrows():
        print(f'  {r["bucket"]:6s}: addon_share={r["addon_share"]*100:.0f}%  c*={r["c_star"]:.2f}')
    print('  HAR-J addon dominates; consider recalibrating GAMMA_AS.')
else:
    print('\nBaseline AS dominates in all buckets (addon share ≤ 50%).')
print(f'\nTest-set regime mix: Low={_rl*100:.0f}%  Mid={_rm*100:.0f}%  High={_rh*100:.0f}%')


## 11. Vega-Weighted Robustness

Spread and RV are in vol-units; P&L impact is proportional to Black-Scholes vega.
Vega proxy: `v(k,T,σ) ≈ φ(d₁) × √T`.
`vega_weighted_shortfall` measures tail losses weighted by economic sensitivity.

In [ ]:
vw_df = compute_vega_weighted_robustness(
    iv_panel, spread_final, targets_df, GRID_COLS, MAT_T_MAP, split_test, H_MAIN
)
vw_df.to_csv(OUTPUT_DIR / f'{NB}_vega_weighted_robustness.csv', index=False)

print('=== VEGA-WEIGHTED ROBUSTNESS (test set) ===')
print(vw_df.round(4).to_string(index=False))


## 12. Summary

In [ ]:
SEP = '=' * 70
print(SEP)
print('NOTEBOOK D — SSVI SURFACE RISK ENGINE — FINAL SUMMARY')
print(SEP)
print(f'''
RESEARCH QUESTION
-----------------
Can localized RV of the SSVI IV surface improve volatility-risk-adjusted
option market-making spreads beyond a baseline Avellaneda-style rule?

SAMPLE
------
  IV panel   : {iv_panel.index[0].date()} → {iv_panel.index[-1].date()}
               ({len(iv_panel)} trading days, {len(GRID_COLS)} grid points)
  Exclusions : {len(_raw) - (_raw["n_obs"] > 0).sum()} dates with n_obs < 100 (see Section 2)
  Train      : {split_train[0].date()} → {split_train[-1].date()} ({len(split_train)} days, 70%)
  Validation : {split_val[0].date()}  → {split_val[-1].date()}  ({len(split_val)} days, 15%)
  Test       : {split_test[0].date()}  → {split_test[-1].date()}  ({len(split_test)} days, 15%)
''')

print('JUMP DETECTION')
print(f'  Method selected: {selected_key}  Rate: {selected_pct:.1f}%  (BPV baseline: {old_jump_pct:.1f}%)')

print('\nFORECAST TABLE (global, h=5, test set):')
_cols = ['model', 'n', 'MAE_log', 'RMSE_log', 'QLIKE', 'DM_stat_vs_HAR', 'DM_pval_vs_HAR']
print(metrics_df[metrics_df['target'] == 'global']
      [[c for c in _cols if c in metrics_df.columns]].round(4).to_string(index=False))

print('\nAS PARAMETERS:')
print(f'  GAMMA_AS={GAMMA_AS}  KAPPA_K={KAPPA_K}  DT=1/252  (uncalibrated proxies)')

print('\nc* CALIBRATION (val set, ES₉₅):')
print(calib_df.round(3).to_string())

print('\nBACKTEST COVERAGE (test set):')
print(bt_df[['bucket', 'coverage', 'exceedance', 'mean_AS', 'mean_addon', 'addon_share', 'n']]
      .round(4).to_string(index=False))

print('\nVEGA-WEIGHTED ROBUSTNESS:')
print(vw_df[['maturity', 'coverage_unweighted', 'vega_weighted_shortfall', 'n']]
      .round(4).to_string(index=False))

print(f'''
LIMITATIONS
-----------
1. AS spread is a heuristic proxy; GAMMA_AS, KAPPA_K are uncalibrated.
2. HMM Viterbi path is retrospective — not suitable for live regime detection.
3. SSVI calibrated end-of-day; no intraday surface updates.
4. No inventory path modelled (full AS requires net delta tracking).
5. c* estimated on a single 15% validation window; rolling-ES would be more robust.
''')
print(SEP); print('Done.'); print(SEP)
